<a href="https://colab.research.google.com/github/Faiq-danZ/worksafe-ai/blob/main/inference/predict_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. MOUNT DRIVE

In [26]:
from google.colab import drive
drive.mount('/content/drive')

path_model = '/content/drive/MyDrive/Data/Models/'
print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## 2. IMPORT Library

In [27]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow import keras
print(f"TF: {tf.__version__}")

TF: 2.20.0


## 3. DEFINISI CUSTOM LOSS (WAJIB ADA SEBELUM LOAD MODEL)

In [28]:
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true    = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true_oh = tf.cast(tf.one_hot(y_true, depth=3), tf.float32)
        y_pred    = tf.cast(y_pred, tf.float32)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce           = -y_true_oh * tf.math.log(y_pred)
        p_t          = tf.reduce_sum(y_true_oh * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        loss         = self.alpha * focal_weight * tf.reduce_sum(ce, axis=-1)
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg

print("FocalLoss OK.")

FocalLoss OK.


## 4. LOAD MODEL & ARTIFACTS

In [29]:
model = keras.models.load_model(
    path_model + 'worksafe_model_v1.keras',
    custom_objects={'FocalLoss': FocalLoss}
)

with open(path_model + 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(path_model + 'imputer.pkl', 'rb') as f:
    imputer = pickle.load(f)

with open(path_model + 'feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

LABEL_MAP = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}
print("Model & artifacts loaded.")
print(f"Jumlah fitur: {len(feature_cols)}")

Model & artifacts loaded.
Jumlah fitur: 50


##  5. FUNGSI PREPROCESSING INPUT

In [30]:
def preprocess_input(input_dict):
    # Buat DataFrame dengan kolom sesuai urutan feature_cols
    df = pd.DataFrame([input_dict])

    # Tambah kolom yang tidak ada sebagai NaN
    for col in feature_cols:
        if col not in df.columns:
            df[col] = np.nan

    # Pastikan urutan kolom sama persis dengan saat training
    df = df[feature_cols]

    # Imputation — pakai DataFrame agar nama kolom terjaga
    df_imp = pd.DataFrame(
        imputer.transform(df),
        columns=feature_cols
    )

    # Scaling — pakai DataFrame agar scaler bisa kenali nama kolom
    df_scaled = pd.DataFrame(
        scaler.transform(df_imp),
        columns=feature_cols
    )

    return df_scaled.values.astype(np.float32)

print("preprocess_input OK.")22223

preprocess_input OK.


## 6. FUNGSI PREDIKSI

In [31]:
def predict_risk(input_dict):
    X     = preprocess_input(input_dict)
    proba = model.predict(X, verbose=0)[0]

    kelas = int(np.argmax(proba))
    label = LABEL_MAP[kelas]

    return {
        'risk_label'    : label,
        'risk_class'    : kelas,
        'confidence'    : round(float(proba[kelas]) * 100, 2),
        'probabilities' : {
            'Low Risk'   : round(float(proba[0]) * 100, 2),
            'Medium Risk': round(float(proba[1]) * 100, 2),
            'High Risk'  : round(float(proba[2]) * 100, 2),
        }
    }

In [32]:
print(feature_cols)

['act_repairing_and_maintaining_mechanical_equipment', 'act_controlling_machines_and_processes', 'skl_operation_and_control', 'skl_equipment_maintenance', 'skl_troubleshooting', 'skl_repairing', 'skl_operations_monitoring', 'act_handling_and_moving_objects', 'act_inspecting_equipment,_structures,_or_materials', 'skl_equipment_selection', 'act_operating_vehicles,_mechanized_devices,_or_equipment', 'act_performing_general_physical_activities', 'act_repairing_and_maintaining_electronic_equipment', 'skl_quality_control_analysis', 'skl_speaking', 'skl_active_listening', 'skl_writing', 'skl_reading_comprehension', 'skl_social_perceptiveness', 'skl_active_learning', 'skl_persuasion', 'act_drafting,_laying_out,_and_specifying_technical_devices,_parts,_and_equipment', 'skl_service_orientation', 'skl_critical_thinking', 'skl_negotiation', 'act_establishing_and_maintaining_interpersonal_relationships', 'act_getting_information', 'skl_learning_strategies', 'skl_judgment_and_decision_making', 'act_

## 7. CONTOH INFERENSI

In [33]:
# Ganti key dan value sesuai nama fitur dari feature_cols
# Bisa cek: print(feature_cols) untuk lihat nama fitur

sample = {fc: np.random.uniform(1, 5) for fc in feature_cols}  # dummy random

result = predict_risk(sample)

print("=" * 40)
print("WorkSafe AI - Hasil Prediksi")
print("=" * 40)
print(f"Label      : {result['risk_label']}")
print(f"Kelas      : {result['risk_class']}")
print(f"Confidence : {result['confidence']}%")
print(f"\nProbabilitas:")
for lbl, prob in result['probabilities'].items():
    bar = '█' * int(prob / 5)
    print(f"  {lbl:<14}: {prob:>5}%  {bar}")

WorkSafe AI - Hasil Prediksi
Label      : Medium Risk
Kelas      : 1
Confidence : 50.83%

Probabilitas:
  Low Risk      :  0.52%  
  Medium Risk   : 50.83%  ██████████
  High Risk     : 48.64%  █████████


In [34]:
print("=" * 50)
print("  WorkSafe AI - Cek Risiko Pekerjaan Kamu")
print("=" * 50)
print("Isi nilai untuk setiap skill di bawah ini.")
print("Skala: 1 (sangat rendah) sampai 5 (sangat tinggi)")
print("-" * 50)

def tanya(pertanyaan, nama_fitur):
    while True:
        try:
            val = float(input(f"{pertanyaan} (1-5): "))
            if 1.0 <= val <= 5.0:
                return nama_fitur, val
            print("  Input harus antara 1 sampai 5.")
        except ValueError:
            print("  Masukkan angka saja.")

pertanyaan_list = [
    # (pertanyaan yang tampil, nama fitur di model)
    ("Kemampuan memperbaiki/merawat mesin",          "act_repairing_and_maintaining_mechanical_equipment"),
    ("Kemampuan mengoperasikan mesin/kendaraan",     "act_operating_vehicles,_mechanized_devices,_or_equipment"),
    ("Kemampuan kontrol peralatan",                  "skl_operation_and_control"),
    ("Kemampuan perbaikan peralatan",                "skl_equipment_maintenance"),
    ("Kemampuan troubleshooting",                    "skl_troubleshooting"),
    ("Kemampuan berpikir kritis",                    "skl_critical_thinking"),
    ("Kemampuan belajar mandiri/aktif",              "skl_active_learning"),
    ("Kemampuan memecahkan masalah kompleks",        "skl_complex_problem_solving"),
    ("Kemampuan analisis sistem",                    "skl_systems_analysis"),
    ("Kemampuan komunikasi/berbicara",               "skl_speaking"),
    ("Kemampuan menulis",                            "skl_writing"),
    ("Kemampuan membaca & memahami teks",            "skl_reading_comprehension"),
    ("Kemampuan mengambil keputusan",                "skl_judgment_and_decision_making"),
    ("Kemampuan manajemen waktu",                    "skl_time_management"),
    ("Kemampuan kerja dengan komputer",              "act_working_with_computers"),
    ("Kemampuan analisis data",                      "act_analyzing_data_or_information"),
    ("Kemampuan perencanaan & organisasi",           "act_organizing,_planning,_and_prioritizing_work"),
    ("Kemampuan bernegosiasi",                       "skl_negotiation"),
    ("Kemampuan koordinasi tim",                     "skl_coordination"),
    ("Kemampuan pemantauan proses/lingkungan",       "act_monitoring_processes,_materials,_or_surroundings"),
]

# Kumpulkan jawaban
sample_input = {}
for pertanyaan, fitur in pertanyaan_list:
    fitur_key, nilai = tanya(pertanyaan, fitur)
    sample_input[fitur_key] = nilai

# Fitur yang tidak diisi → isi dengan nilai tengah (3.0)
for col in feature_cols:
    if col not in sample_input:
        sample_input[col] = 3.0

# Prediksi
print("\nMemproses...")
result = predict_risk(sample_input)

print("\n" + "=" * 50)
print("  HASIL ANALISIS RISIKO PEKERJAAN KAMU")
print("=" * 50)
print(f"  Kategori Risiko : {result['risk_label']}")
print(f"  Confidence      : {result['confidence']}%")
print("-" * 50)
print("  Probabilitas tiap kategori:")
for lbl, prob in result['probabilities'].items():
    bar = '█' * int(prob / 5)
    print(f"  {lbl:<14}: {prob:>6}%  {bar}")
print("=" * 50)

# Interpretasi otomatis
if result['risk_class'] == 0:
    print("\n  Pekerjaan kamu tergolong AMAN dari risiko otomasi.")
    print("  Profil kompetensi kamu didominasi kemampuan kognitif")
    print("  dan sosial yang sulit digantikan mesin.")
elif result['risk_class'] == 1:
    print("\n  Pekerjaan kamu memiliki risiko SEDANG.")
    print("  Ada sebagian tugas yang berpotensi terotomasi,")
    print("  namun masih banyak aspek yang membutuhkan manusia.")
else:
    print("\n  Pekerjaan kamu memiliki risiko TINGGI terkena otomasi.")
    print("  Disarankan untuk mulai mengembangkan skill kognitif")
    print("  dan digital agar tetap relevan di pasar kerja.")

  WorkSafe AI - Cek Risiko Pekerjaan Kamu
Isi nilai untuk setiap skill di bawah ini.
Skala: 1 (sangat rendah) sampai 5 (sangat tinggi)
--------------------------------------------------
Kemampuan memperbaiki/merawat mesin (1-5): 2
Kemampuan mengoperasikan mesin/kendaraan (1-5): 
  Masukkan angka saja.
Kemampuan mengoperasikan mesin/kendaraan (1-5): 2
Kemampuan kontrol peralatan (1-5): 2
Kemampuan perbaikan peralatan (1-5): 2
Kemampuan troubleshooting (1-5): 2
Kemampuan berpikir kritis (1-5): 
  Masukkan angka saja.
Kemampuan berpikir kritis (1-5): 
  Masukkan angka saja.
Kemampuan berpikir kritis (1-5): 3
Kemampuan belajar mandiri/aktif (1-5): 3
Kemampuan memecahkan masalah kompleks (1-5): 3
Kemampuan analisis sistem (1-5): 3
Kemampuan komunikasi/berbicara (1-5): 3
Kemampuan menulis (1-5): 3
Kemampuan membaca & memahami teks (1-5): 2
Kemampuan mengambil keputusan (1-5): 4
Kemampuan manajemen waktu (1-5): 5
Kemampuan kerja dengan komputer (1-5): 5
Kemampuan analisis data (1-5): 4
Kemamp

## 8. BATCH INFERENCE (OPSIONAL)

In [36]:
# Contoh prediksi dari CSV yang sudah ada

path_processed = '/content/drive/MyDrive/Data/Data/Processed/'
df_infer = pd.read_csv(path_processed + 'test_data.csv').head(10)

results = []
for _, row in df_infer.iterrows():
    inp = {col: row[col] for col in feature_cols if col in df_infer.columns}
    res = predict_risk(inp)
    results.append({
        'risk_label' : res['risk_label'],
        'confidence' : res['confidence'],
        'actual'     : int(row['risk_label']) if 'risk_label' in row else None
    })

df_results = pd.DataFrame(results)
df_results['actual_label'] = df_results['actual'].map(LABEL_MAP)
print("Hasil batch inference (10 sampel pertama):")
display(df_results[['actual_label', 'risk_label', 'confidence']])

Hasil batch inference (10 sampel pertama):


,actual_label,risk_label,confidence
0,High Risk,Medium Risk,100.00
1,Low Risk,Medium Risk,99.80
2,High Risk,Medium Risk,100.00
3,Medium Risk,Medium Risk,100.00
4,Medium Risk,Medium Risk,100.00
5,High Risk,Medium Risk,100.00
6,High Risk,Medium Risk,100.00
7,Low Risk,Medium Risk,90.47
8,Low Risk,Medium Risk,99.99
9,Low Risk,Medium Risk,87.52
